# 10. Tournament Simulation & Monte Carlo Engine

This notebook demonstrates the Monte Carlo simulation engine for the FIFA World Cup.
It uses the champion machine learning model (XGBoost / LightGBM) trained on international match history,
FIFA rankings, Elo ratings, and squad quality features to simulate the entire tournament from
group stages through the final.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
from src.simulate_tournament import TournamentSimulator

print('Imports successful.')

## 1. Initialize Tournament Simulator
Loads model checkpoint (`models/best_model.pkl`) and latest team profiles (Elo, FIFA Rankings, FIFA Attributes).

In [ ]:
simulator = TournamentSimulator(
    model_path='../models/best_model.pkl',
    elo_path='../data/interim/elo_clean.csv',
    rankings_path='../data/interim/rankings_clean.csv',
    match_features_path='../data/interim/match_features_clean.csv'
)
print(f'Loaded {len(simulator.all_teams)} tournament teams across {len(simulator.groups)} groups.')

## 2. Simulate Single Tournament Run
Simulate one complete tournament bracket and inspect outcomes.

In [ ]:
single_res = simulator.simulate_single_tournament()
print('Group Advancers (1st & 2nd):')
for grp, (first, second) in single_res['group_advancers'].items():
    print(f'  Group {grp}: 1st: {first}, 2nd: {second}')

print(f'\nQuarterfinalists: {single_res["qf"]}')
print(f'Semifinalists: {single_res["sf"]}')
print(f'Finalists: {single_res["final"]}')
print(f'🏆 Champion: {single_res["champion"]}')
print(f'🥈 Runner-Up: {single_res["runner_up"]}')
print(f'🥉 3rd Place: {single_res["third"]}')

## 3. Run Monte Carlo Simulation (10,000 Iterations)
Simulate 10,000 World Cup tournaments to compute win probabilities at each stage.

In [ ]:
summary_df, meta = simulator.run_monte_carlo(num_simulations=10000, seed=42)
summary_df.head(15)

## 4. Visualize Tournament Win Probabilities

In [ ]:
top10 = summary_df.head(10)
fig = px.bar(
    top10,
    x='team',
    y='champion_pct',
    color='champion_pct',
    labels={'team': 'Country', 'champion_pct': 'Champion Probability (%)'},
    title='Top 10 FIFA World Cup Favorites (10,000 Monte Carlo Simulations)',
    text='champion_pct'
)
fig.update_layout(template='plotly_white')
fig.show()